# Project Milestone 3

In [1]:
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# read from the html file
with open('List of countries by literacy rate - Wikipedia.html','r',encoding='utf-8') as file:
    html = file.read()

# parse with beautifulsoup
soup = BeautifulSoup(html, 'html.parser')

# find the table
table = soup.find('table', {'class':'wikitable'})

#read the table as dataframe
df = pd.read_html(str(table))[0]

df.head()

Country Youth (15 to 24)          Adult (25+)          \
            Country             Rate     Year        Rate    Year   
0     Afghanistan *             65.0  2020[4]        31.7  2011.0   
1         Albania *             99.2     2012        97.2  2012.0   
2         Algeria *             93.8     2008        75.1  2008.0   
3  American Samoa *             97.7     1980        97.3  1980.0   
4         Andorra *              NaN      NaN         NaN     NaN   

  Elderly (65+)         Youth Gender Parity Index          
           Rate    Year                      Rate    Year  
0          20.3  2011.0                       0.5  2011.0  
1          86.9  2012.0                       1.0  2012.0  
2          19.5  2008.0                       1.0  2008.0  
3          92.7  1980.0                       1.0  1980.0  
4           NaN     NaN                       NaN     NaN

### Step 1: Flatten multi-level columns

The columns are in multi-level. Flattening the columns will simplify table structure and make is easier to manipulate.


In [3]:
#combine column names 
df.columns = [
    f"{outer} {inner}".strip()
    for outer, inner in df.columns
]

In [4]:
df.head()

,Country Country,Youth (15 to 24) Rate,Youth (15 to 24) Year,Adult (25+) Rate,Adult (25+) Year,Elderly (65+) Rate,Elderly (65+) Year,Youth Gender Parity Index Rate,Youth Gender Parity Index Year
0,Afghanistan *,65.0,2020[4],31.7,2011.0,20.3,2011.0,0.5,2011.0
1,Albania *,99.2,2012,97.2,2012.0,86.9,2012.0,1.0,2012.0
2,Algeria *,93.8,2008,75.1,2008.0,19.5,2008.0,1.0,2008.0
3,American Samoa *,97.7,1980,97.3,1980.0,92.7,1980.0,1.0,1980.0
4,Andorra *,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 2: Rename column names

Renaming column names improves readability, saves typing time, and gives consistent pattern.


In [5]:
#rename column names manually
df.rename(columns={
    'Country Country': 'Country',
    'Youth (15 to 24) Rate': 'Youth Rate',
    'Youth (15 to 24) Year': 'Youth Year',
    'Adult (25+) Rate': 'Adult Rate',
    'Adult (25+) Year': 'Adult Year',
    'Elderly (65+) Rate': 'Elderly Rate',
    'Elderly (65+) Year': 'Elderly Year',
    'Youth Gender Parity Index Rate': 'Parity Rate',
    'Youth Gender Parity Index Year': 'Parity Year'
}, inplace=True)

df.head()

,Country,Youth Rate,Youth Year,Adult Rate,Adult Year,Elderly Rate,Elderly Year,Parity Rate,Parity Year
0,Afghanistan *,65.0,2020[4],31.7,2011.0,20.3,2011.0,0.5,2011.0
1,Albania *,99.2,2012,97.2,2012.0,86.9,2012.0,1.0,2012.0
2,Algeria *,93.8,2008,75.1,2008.0,19.5,2008.0,1.0,2008.0
3,American Samoa *,97.7,1980,97.3,1980.0,92.7,1980.0,1.0,1980.0
4,Andorra *,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 3: Clean country names

Clean country names by removing * and footnote marks that are imported from html.

For example, 'Greenland *[8]' should be cleaned to 'Greenland'

In [6]:
# clean using regex
df['Country'] = (
    df['Country'].astype(str).str.replace('*','',regex=False) #remove asterisks
    .str.replace(r"\[.*?\]", "", regex=True)                  #remove brackets
    .str.strip()                                              #remove spaces
)

In [7]:
# validate
df[df['Country'] == 'Greenland']

,Country,Youth Rate,Youth Year,Adult Rate,Adult Year,Elderly Rate,Elderly Year,Parity Rate,Parity Year
83,Greenland,100.0,2015,NaN,NaN,NaN,NaN,1.0,2015.0


### Step 4: Clean all Rate columns

Ensure all Rate columns are cleaned by converting to float type and removing footnote marks.

In [8]:
#all rate columns
rate_cols = ['Youth Rate', 'Adult Rate', 'Elderly Rate', 'Parity Rate']

for col in rate_cols:
    df[col] = (df[col].astype(str).str.replace(r"\[.*?\]", "", regex=True)   # remove footnotes
               .str.strip()
              )
    df[col] = pd.to_numeric(df[col],errors='coerce')

In [9]:
# Validate
df['Youth Rate'].unique()

array([ 65. ,  99.2,  93.8,  97.7,   nan,  77.4,  99.1,  99.5,  99.8,
        96.6,  99.9,  97. ,  94.9,  77.6,  52.5,  87.3,  99.4,  99.7,
        97.9,  50.1,  79.6,  98.1,  92.2,  80.6,  98.9,  36.4,  30.8,
        98.7,  71.6,  80.9,  85. ,  53. ,  98.8,  93.9,  98. ,  98.2,
        87. ,  93.5,  55. ,  91.3,  88.5,  60.8,  85.7, 100. ,  94.4,
        46.3,  60.4,  96.7,  72.3,  96.1,  95.7,  98.6,  96.3,  86.5,
        92.5,  86.6,  49.1,  99.6,  76.8,  72.9,  99.3,  49.4,  98.5,
        56.1,  70.5,  84.8,  39.8,  66.4,  74.5,  99. ,  97.6,  67.9,
        95.9,  85.1,  69.5,  57. ,  36.7,  65.8,  85.8,  79.5,  84.3,
        96.2,  83.7,  95. ,  95.3,  97.1,  77. ,  88.7,  90.4])

### Step 5: Clean all Year columns

Ensure all Year columns are cleaned by removing footnotes and convert to int

In [10]:
year_cols = ['Youth Year', 'Adult Year', 'Elderly Year', 'Parity Year']

for col in year_cols:
    df[col] = (
        df[col].astype(str).str.replace(r"\[.*?\]", "", regex=True)   # remove footnotes
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

In [11]:
df['Youth Year'].unique()

<IntegerArray>
[2020, 2012, 2008, 1980, <NA>, 2014, 1984, 2016, 2011, 2010, 2001, 2019, 2009,
 1991, 2015, 2013, 2007, 2018, 2017, 1982, 1990, 2006, 1983, 2004, 2005, 2002,
 1970]
Length: 27, dtype: Int64

### Step 6: Drop rows with too much missing data

Remove rows where key literacy rate values are missing. It will reduce noises and avoid misleading stats.

In [12]:
#drop where youth rate, adult rate are missing
df.dropna(subset=["Youth Rate", "Adult Rate"], inplace=True)

df

,Country,Youth Rate,Youth Year,Adult Rate,Adult Year,Elderly Rate,Elderly Year,Parity Rate,Parity Year
0,Afghanistan,65.0,2020,31.7,2011,20.3,2011,0.5,2011
1,Albania,99.2,2012,97.2,2012,86.9,2012,1.0,2012
2,Algeria,93.8,2008,75.1,2008,19.5,2008,1.0,2008
3,American Samoa,97.7,1980,97.3,1980,92.7,1980,1.0,1980
5,Angola,77.4,2014,66.0,2014,27.0,2014,0.8,2014
...,...,...,...,...,...,...,...,...,...
227,Venezuela,98.8,2016,97.1,2016,88.1,2016,1.0,2016
228,Vietnam,97.1,2009,95.8,2019,76.9,2009,1.0,2009
231,Yemen,77.0,2004,54.1,2004,13.7,2004,0.7,2004
232,Zambia,88.7,2010,83.0,2010,52.3,2010,0.9,2010


### Step 7: Create a new column - Youth-Adult gap

Create a new column called 'Youth-Adult Gap' to show the difference between youth and adult literacy rates for each country. This will highlight generational gaps and show whether younger people are significantly more literate than adults.

In [13]:
# create a new columns
df['Youth-Adult Gap'] = df['Youth Rate'] - df['Adult Rate']

In [14]:
df.head()

,Country,Youth Rate,Youth Year,Adult Rate,Adult Year,Elderly Rate,Elderly Year,Parity Rate,Parity Year,Youth-Adult Gap
0,Afghanistan,65.0,2020,31.7,2011,20.3,2011,0.5,2011,33.3
1,Albania,99.2,2012,97.2,2012,86.9,2012,1.0,2012,2.0
2,Algeria,93.8,2008,75.1,2008,19.5,2008,1.0,2008,18.7
3,American Samoa,97.7,1980,97.3,1980,92.7,1980,1.0,1980,0.4
5,Angola,77.4,2014,66.0,2014,27.0,2014,0.8,2014,11.4


### Final Output

In [15]:
df.head(25)

,Country,Youth Rate,Youth Year,Adult Rate,Adult Year,Elderly Rate,Elderly Year,Parity Rate,Parity Year,Youth-Adult Gap
0,Afghanistan,65.0,2020,31.7,2011,20.3,2011,0.50,2011,33.3
1,Albania,99.2,2012,97.2,2012,86.9,2012,1.00,2012,2.0
2,Algeria,93.8,2008,75.1,2008,19.5,2008,1.00,2008,18.7
3,American Samoa,97.7,1980,97.3,1980,92.7,1980,1.00,1980,0.4
5,Angola,77.4,2014,66.0,2014,27.0,2014,0.80,2014,11.4
6,Anguilla,99.1,1984,95.4,1984,88.0,1984,1.00,1984,3.7
8,Argentina,99.5,2016,99.1,2016,97.9,2016,1.00,2016,0.4
9,Armenia,99.8,2011,99.7,2011,98.9,2011,1.00,2011,0.1
10,Aruba,99.1,2010,96.8,2010,88.9,2010,1.00,2010,2.3
13,Azerbaijan,99.9,2016,99.8,2016,98.4,2016,1.00,2016,0.1


In [18]:
df['Country'] = df["Country"].str.lower()
df.to_csv("website.csv",index=False)